# Planck Scans — Batch Inference
Runs all real scans from `planck_scans/npy_scaled/` through the full pipeline
(backbone → transformer → coarse matching → morph decoder → Sinkhorn → LGR).
No ground truth — qualitative only. Dummy `gt_z=zeros` and `transform=eye`.

**Workflow:**
1. Run all scans once (inference + Chamfer) → stored in `results`
2. Summary table + bar chart
3. Pick a scan by name → open two O3D windows (before / after alignment)

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
SNAPSHOT  = 'epoch-40.pth.tar'  # checkpoint filename inside output/.../snapshots/
DEVICE    = 'cuda'

# Preprocessing for planck_scans/npy_scaled — scans are already in UHM meter-scale units.
# Only centering (subtract centroid) is applied. Flip Z if your scanner Z points away from face.
FLIP_Z  = False   # set True if scans arrive with Z pointing away from face
CENTER  = True    # subtract centroid to place scan at origin
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import os, sys, glob

EXP_DIR  = os.path.dirname(os.path.abspath('__file__'))
ROOT_DIR = os.path.dirname(os.path.dirname(EXP_DIR))
sys.path.insert(0, EXP_DIR)
sys.path.insert(0, ROOT_DIR)
os.chdir(EXP_DIR)

import torch
import numpy as np
import plotly.graph_objects as go
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Image, display

from geotransformer.modules.ops.transformation import apply_transform

from config_dowsampled import make_cfg
from dataset import train_valid_data_loader
from model import create_model

print('EXP_DIR :', EXP_DIR)
print('ROOT_DIR:', ROOT_DIR)

In [ ]:
cfg = make_cfg()

_, _, neighbor_limits = train_valid_data_loader(
    cfg, distributed=False, val_aug_scale=1.0, val_aug_subsample=1.0
)
print('neighbor_limits:', neighbor_limits)

SNAP_DIR = os.path.join(
    ROOT_DIR, 'output',
    'geotransformer.facesdownsampledfixed.stage4.gse.k3.max.oacl.stage2.sinkhorn.eli',
    'snapshots',
)
ckpt_path = os.path.join(SNAP_DIR, SNAPSHOT)

model = create_model(cfg).to(DEVICE)
model.neighbor_limits = neighbor_limits

ckpt = torch.load(ckpt_path, map_location=DEVICE)
sd   = ckpt.get('model', ckpt)
sd   = {k.replace('module.', ''): v for k, v in sd.items()}

missing = set(model.state_dict()) - set(sd)
extra   = set(sd) - set(model.state_dict())
if missing: print(f'[warn] missing keys (random init): {len(missing)}')
if extra:   print(f'[info] extra keys ignored         : {len(extra)}')

model.load_state_dict(sd, strict=False)
model.eval()

print(f'Loaded: {SNAPSHOT}  (epoch {ckpt.get("epoch", "?")})')
print(f'Keys matched: {len(set(sd) & set(model.state_dict()))} / {len(model.state_dict())}')

with torch.no_grad():
    mean_ref = model.generate_reference_geometry(torch.zeros(32, 36, device=DEVICE))
print(f'Mean ref: {mean_ref.shape}')

In [ ]:
def chamfer_distance(A, B, device=DEVICE, single_sided=False):
    """Symmetric Chamfer distance (mean of bidirectional NN distances, metres)."""
    if isinstance(A, np.ndarray):
        A = torch.from_numpy(A).float().to(device)
    if isinstance(B, np.ndarray):
        B = torch.from_numpy(B).float().to(device)
    chunk = 2048
    dists_AB = [torch.cdist(A[i:i+chunk], B).min(dim=1).values for i in range(0, A.shape[0], chunk)]
    dists_BA = [torch.cdist(B[i:i+chunk], A).min(dim=1).values for i in range(0, B.shape[0], chunk)]
    if single_sided:
        chamfer =  torch.stack(dists_AB).mean().item()
        std = torch.stack(dists_AB).std().item()
    else: 
        chamfer = 0.5 * (torch.cat(dists_AB).mean().item() + torch.cat(dists_BA).mean().item())
        std = 0.5 * (torch.cat(dists_AB).std().item() + torch.cat(dists_BA).std().item())
    return chamfer , std

def hausdorff_distance(A, B, device=DEVICE , single_sided=False):
    """Symmetric Hausdorff distance (max of bidirectional NN distances, metres)."""
    if isinstance(A, np.ndarray):
        A = torch.from_numpy(A).float().to(device)
    if isinstance(B, np.ndarray):
        B = torch.from_numpy(B).float().to(device)
    chunk = 2048
    dists_AB = [torch.cdist(A[i:i+chunk], B).min(dim=1).values for i in range(0, A.shape[0], chunk)]
    dists_BA = [torch.cdist(B[i:i+chunk], A).min(dim=1).values for i in range(0, B.shape[0], chunk)]
    if single_sided:
        hausdorff = max(torch.cat(dists_AB).max().item(), torch.cat(dists_BA).max().item())
    else:
        hausdorff = max(torch.cat(dists_AB).max().item(), torch.cat(dists_BA).max().item())
    return hausdorff



def preprocess_scan(path, flip_z=FLIP_Z, center=CENTER, device=DEVICE):
    raw = np.load(path)
    pts = torch.from_numpy(raw[:, :3].astype(np.float32)).to(device)
    if flip_z:
        pts[:, 2] *= -1
    if center:
        pts -= pts.mean(dim=0, keepdim=True)
    return pts


def run_inference(src_pts, mean_ref, model, device=DEVICE):
    n_ref = mean_ref.shape[0]
    n_src = src_pts.shape[0]
    data_dict = {
        'points'   : torch.cat([mean_ref, src_pts], dim=0),
        'lengths'  : torch.tensor([n_ref, n_src], dtype=torch.int64),
        'features' : torch.ones(n_ref + n_src, 1, device=device),
        'gt_z'     : torch.zeros(32, 36, device=device),
        'transform': torch.eye(4, device=device),
    }
    with torch.no_grad():
        out = model(data_dict)
    return out


def pcd_trace(pts, color, name, size=2, opacity=0.7):
    if isinstance(pts, torch.Tensor):
        pts = pts.detach().cpu().numpy()
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity),
        name=name,
    )


def show_pcd(traces, title='', height=600):
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title, height=height,
        scene=dict(aspectmode='data'),
        legend=dict(itemsizing='constant'),
        margin=dict(l=0, r=0, b=0, t=40),
    )
    fig.show()

---
## Run all scans (inference + Chamfer)

In [ ]:
SCANS_DIR = os.path.join(ROOT_DIR, 'planck_scans', 'npy_scaled')
scan_paths = sorted(
    glob.glob(os.path.join(SCANS_DIR, '*.npy')),
    key=lambda p: int(''.join(filter(str.isdigit, os.path.basename(p))) or '0'),
)
print(f'Found {len(scan_paths)} scans in {SCANS_DIR}')
for p in scan_paths:
    print(' ', os.path.basename(p))

In [ ]:
results = {}  # name -> result dict

for scan_path in scan_paths:
    scan_name = os.path.basename(scan_path)
    try:
        src_pts = preprocess_scan(scan_path)
        out     = run_inference(src_pts, mean_ref, model)

        morphed_ref = out['morphed_full']
        est_tf      = out['estimated_transform']
        src_model   = out['src_points']
        src_aligned = apply_transform(src_model, est_tf)

        cd, cd_std = chamfer_distance(src_aligned, morphed_ref, single_sided=True)
        cd_mean, cd_mean_std = chamfer_distance(src_aligned, mean_ref, single_sided=True)
        hausadorff = hausdorff_distance(src_aligned, morphed_ref, single_sided=True)
        hausadorff_mean = hausdorff_distance(src_aligned, mean_ref, single_sided=True)

        results[scan_name] = {
            'name'        : scan_name,
            'src_raw'     : src_pts.cpu().numpy(),
            'src_aligned' : src_aligned.cpu().numpy(),
            'morphed_ref' : morphed_ref.cpu().numpy(),
            'mean_ref'    : mean_ref.cpu().numpy(),
            'chamfer'     : cd,
            'chamfer_std' : cd_std,
            'chamfer_mean': cd_mean,
            'chamfer_mean_std': cd_mean_std,
            'n_src'       : src_pts.shape[0],
            'error'       : None,
        }
        print(f'  {scan_name:20s}  {src_pts.shape[0]:5d} pts  | CD={cd:.4f} m  |  dist_std={cd_std:.4f} m | CD_mean_ref={cd_mean:.4f} m |  dist_mean_std={cd_mean_std:.4f} m | Hausdorff={hausadorff:.4f} m | Hausdorff_mean={hausadorff_mean:.4f} m')

    except Exception as e:
        print(f'  {scan_name:20s}  ERROR: {e}')
        results[scan_name] = {'name': scan_name, 'error': str(e), 'chamfer': float('nan'), 'chamfer_std': float('nan'), 'chamfer_mean': float('nan'), 'chamfer_mean_std': float('nan')}

print(f'\nDone — {sum(r["error"] is None for r in results.values())}/{len(results)} succeeded')

In [ ]:
import pandas as pd

rows = []
for k, r in results.items():
    rows.append({
        'name': r.get('name', k),
        'n_src': r.get('n_src', np.nan),
        'chamfer': r.get('chamfer', np.nan),
        'chamfer_mean': r.get('chamfer_mean', np.nan),
        'error': r.get('error', None),
    })

df_results = pd.DataFrame(rows).set_index('name').sort_index()
df_results

---
## Summary statistics

In [ ]:
ok     = [r for r in results.values() if r['error'] is None]
cd_vals = np.array([r['chamfer'] for r in ok])
cd_mean_vals = np.array([r['chamfer_mean'] for r in ok])
diff_vals = cd_mean_vals - cd_vals
scaling_factor = 100  # to convert metres to centimetres (UHM model is ~2.5m, real heads are ~25cm)
print('=' * 162)
print(f"UHM model is height is ~2.5 this scaled ~1/10 to real heads in cm scale, for meausers in mm use scale factor {scaling_factor}") 
print('=' * 162)
print(f"{'Scan':<22}  {'n_pts':>6}  {'CD (uhm)':>12} {'CD (mm)':>12} {'CD Mean (uhm)':>12} {'CD Mean (mm)':>12}  {'Diff (uhm)':>10} {'Diff (mm)':>10}")
print('-' * 162)
for r in results.values():
    if r['error'] is None:
        diff = r['chamfer_mean'] - r['chamfer']
        chamfer_mm = r['chamfer'] * scaling_factor
        chamfer_mean_mm = r['chamfer_mean'] * scaling_factor
        diff_mm = diff * scaling_factor
        print(f"{r['name']:<22}  {r['n_src']:>6}  {r['chamfer']:>12.4f}  {chamfer_mm:>12.4f}  {r['chamfer_mean']:>12.4f}  {chamfer_mean_mm:>12.4f}  {diff:>10.4f}  {diff_mm:>10.4f}")
    else:
        print(f"{r['name']:<22}  {'ERROR':>20}  {'':>12}  {r['error']}")

print('=' * 162)
print(f"{'mean':<22}  {'':>6}  {cd_vals.mean():>12.4f} {cd_vals.mean()*scaling_factor:>12.4f}  {cd_mean_vals.mean():>12.4f} {cd_mean_vals.mean()*scaling_factor:>12.4f}  {diff_vals.mean():>10.4f} {diff_vals.mean()*scaling_factor:>10.4f}")
print(f"{'median':<22}  {'':>6}  {np.median(cd_vals):>12.4f} {np.median(cd_vals)*scaling_factor:>12.4f}  {np.median(cd_mean_vals):>12.4f} {np.median(cd_mean_vals)*scaling_factor:>12.4f}  {np.median(diff_vals):>10.4f} {np.median(diff_vals)*scaling_factor:>10.4f}")
print(f"{'std':<22}  {'':>6}  {cd_vals.std():>12.4f} {cd_vals.std()*scaling_factor:>12.4f}  {cd_mean_vals.std():>12.4f} {cd_mean_vals.std()*scaling_factor:>12.4f}  {diff_vals.std():>10.4f} {diff_vals.std()*scaling_factor:>10.4f}")
print(f"{'min':<22}  {'':>6}  {cd_vals.min():>12.4f} {cd_vals.min()*scaling_factor:>12.4f}  {cd_mean_vals.min():>12.4f} {cd_mean_vals.min()*scaling_factor:>12.4f}  {diff_vals.min():>10.4f} {diff_vals.min()*scaling_factor:>10.4f}")
print(f"{'max':<22}  {'':>6}  {cd_vals.max():>12.4f} {cd_vals.max()*scaling_factor:>12.4f}  {cd_mean_vals.max():>12.4f} {cd_mean_vals.max()*scaling_factor:>12.4f}  {diff_vals.max():>10.4f} {diff_vals.max()*scaling_factor:>10.4f}")

In [ ]:
fig = go.Figure()

cd_values = [r['chamfer'] for r in results.values() if r['error'] is None]
cd_mean_values = [r['chamfer_mean'] for r in results.values() if r['error'] is None]

fig.add_trace(go.Histogram(
    x=cd_values,
    name='Chamfer (src_aligned ↔ morphed_ref)',
    opacity=0.6,
    marker_color='steelblue',
    nbinsx=20,
))

fig.add_trace(go.Histogram(
    x=cd_mean_values,
    name='Chamfer Mean (src_aligned ↔ mean_ref)',
    opacity=0.6,
    marker_color='tomato',
    nbinsx=20,
))

fig.update_layout(
    barmode='overlay',
    title='Distribution of Chamfer distances',
    xaxis_title='Chamfer distance (m)',
    yaxis_title='Frequency',
    legend_title_text='Metric',
)

fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(max(6, len(ok) * 0.7), 4))
ax.bar([r['name'] for r in ok], [r['chamfer'] for r in ok],
       color='steelblue', edgecolor='white', linewidth=0.5)
ax.axhline(cd_vals.mean(),     color='tomato',   linestyle='--', linewidth=1.2,
           label=f'mean={cd_vals.mean():.4f} m')
ax.axhline(np.median(cd_vals), color='seagreen', linestyle=':',  linewidth=1.2,
           label=f'median={np.median(cd_vals):.4f} m')
ax.set_xlabel('Scan')
ax.set_ylabel('Chamfer distance (m)')
ax.set_title('Chamfer distance per scan  (src_aligned ↔ morphed_ref)')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=8)
plt.tight_layout()

buf = BytesIO()
fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

In [ ]:
# grouped bar chart: chamfer (aligned↔morphed) vs chamfer_mean (aligned↔mean_ref)
names = [r['name'] for r in ok]
vals = [r['chamfer'] for r in ok]
vals_mean = [r['chamfer_mean'] for r in ok]

mean_cd = np.mean(vals)
median_cd = np.median(vals)
mean_cd_mean = np.mean(vals_mean)
median_cd_mean = np.median(vals_mean)

x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(max(6, len(ok) * 0.7), 4))
ax.bar(x - w/2, vals, width=w, color='steelblue', edgecolor='white', linewidth=0.5, label='Chamfer (aligned ↔ morphed_ref)')
ax.bar(x + w/2, vals_mean, width=w, color='tomato', edgecolor='white', linewidth=0.5, label='Chamfer Mean (aligned ↔ mean_ref)')

ax.axhline(mean_cd,       color='navy',     linestyle='--', linewidth=1.2, label=f'mean chamfer={mean_cd:.4f} m')
ax.axhline(median_cd,     color='seagreen', linestyle=':',  linewidth=1.2, label=f'median chamfer={median_cd:.4f} m')
ax.axhline(mean_cd_mean,  color='darkred',  linestyle='--', linewidth=1.2, label=f'mean chamfer_mean={mean_cd_mean:.4f} m')
ax.axhline(median_cd_mean,color='orange',   linestyle=':',  linewidth=1.2, label=f'median chamfer_mean={median_cd_mean:.4f} m')

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45)
ax.set_xlabel('Scan')
ax.set_ylabel('Chamfer distance (m)')
ax.set_title('Chamfer distances per scan')
ax.legend(fontsize=8)
plt.tight_layout()

buf = BytesIO()
fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

---
## Open3D visualization — pick a scan
Set `VIZ_SCAN` to any scan name from the table above, then run the next cell.
Two inline Plotly figures render in the notebook:
- **Plot 1:** mean ref (blue) + raw scan (red) — *before* alignment
- **Plot 2:** morphed ref (blue) + aligned scan (orange) — *after* alignment

In [ ]:
SCAN_NUM = 15
VIZ_SCAN = f'planck_{SCAN_NUM}.npy'   # ← change this to any scan name from the table above

print('Available scans:')
for name, r in results.items():
    tag = f"CD={r['chamfer']:.4f} m" if r['error'] is None else f"ERROR: {r['error']}"
    marker = '  ◀' if name == VIZ_SCAN else ''
    print(f'  {name:<22}  {tag}{marker}')

In [ ]:
def plot_chamfer_heatmap(src_pts, ref_pts, title='One-sided Chamfer heatmap', colorscale='Turbo', point_size=2, dist_scale=10):
    """
    Compute one-sided Chamfer distance from src_pts to ref_pts and plot src_pts
    colored by nearest-neighbor distance to ref_pts.
    
    Returns:
        chamfer_mean, nn_distances, fig
    """
    if isinstance(src_pts, np.ndarray):
        src = torch.from_numpy(src_pts).float()
    else:
        src = src_pts.detach().float().cpu()

    if isinstance(ref_pts, np.ndarray):
        ref = torch.from_numpy(ref_pts).float()
    else:
        ref = ref_pts.detach().float().cpu()

    chunk = 2048
    nn_dists = []
    for i in range(0, src.shape[0], chunk):
        d = torch.cdist(src[i:i + chunk], ref).min(dim=1).values
        nn_dists.append(d)
    nn_dist = torch.cat(nn_dists, dim=0)*dist_scale

    chamfer_one_sided = nn_dist.mean().item()
    
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=src[:, 0].numpy(),
                y=src[:, 1].numpy(),
                z=src[:, 2].numpy(),
                mode='markers',
                marker=dict(
                    size=point_size,
                    color=nn_dist.numpy(),
                    colorscale=colorscale,
                    showscale=True,
                    colorbar=dict(title='NN dist (cm)'),
                    opacity=0.9,
                ),
                name=f'src (CD={chamfer_one_sided:.4f} cm)',
            )
        ]
    )

    full_title = f"{title}<br><span style='font-size:12px;color:gray;'>Chamfer (one-sided) = {chamfer_one_sided:.4f} cm</span>"
    fig.update_layout(
        title=full_title,
        scene=dict(aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(itemsizing='constant'),
    )
    fig.show()

In [ ]:
r = results.get(VIZ_SCAN)
assert r is not None,  f'{VIZ_SCAN!r} not found in results — check the name'
assert r['error'] is None, f'{VIZ_SCAN} failed inference: {r["error"]}'

cd = r['chamfer']
print(f'Visualising: {VIZ_SCAN}  ({r["n_src"]} pts)  CD={cd:.4f} m')

# ── Before alignment ──────────────────────────────────────────────────────────
show_pcd(
    [pcd_trace(r['mean_ref'], 'steelblue', 'mean ref'),
     pcd_trace(r['src_raw'],  'tomato',    f'raw scan ({r["n_src"]} pts)')],
    title=f'{VIZ_SCAN} — Before alignment',
)

# ── After alignment ───────────────────────────────────────────────────────────
show_pcd(
    [pcd_trace(r['morphed_ref'], 'steelblue', 'morphed ref'),
     pcd_trace(r['src_aligned'], 'orange',    f'aligned scan  CD={cd:.4f} m')],
    title=f'{VIZ_SCAN} — After alignment',
)

# ── Morph quality ─────────────────────────────────────────────────────────────
show_pcd(
    [pcd_trace(r['mean_ref'],    'steelblue', 'mean ref'),
     pcd_trace(r['morphed_ref'], 'orange',  'morphed ref (pred z)')],
    title=f'{VIZ_SCAN} — Morph quality',
)

In [ ]:
# Re-run inference for the selected scan to get its per-patch coefficients
rr = results.get(VIZ_SCAN)
assert rr is not None and rr['error'] is None, f"{VIZ_SCAN} is missing or failed."

src_pts_viz = torch.from_numpy(rr['src_raw']).float().to(DEVICE)
out_viz = run_inference(src_pts_viz, mean_ref, model)

z = out_viz['z_coefficients'].detach().cpu().float()
if z.ndim == 3 and z.shape[0] == 1:
    z = z[0]
if z.ndim != 2:
    raise ValueError(f"Unexpected z_coefficients shape: {tuple(z.shape)}")

# Ensure rows are patches and columns are coefficients
if z.shape[0] != 32 and z.shape[1] == 32:
    z = z.T

n_patches, n_coeffs = z.shape
assert n_patches == 32, f"Expected 32 patches, got {n_patches}"
assert n_coeffs == 36, f"Expected 36 coefficients, got {n_coeffs}"

fig, axes = plt.subplots(8, 4, figsize=(20, 24), sharex=True, sharey=True)
axes = axes.ravel()
x = np.arange(n_coeffs)

for i in range(n_patches):
    ax = axes[i]
    ax.bar(x, z[i].numpy(), color='steelblue', width=0.8)
    ax.set_title(f'Patch {i+1}', fontsize=9)
    ax.axhline(0.0, color='black', linewidth=0.6)
    ax.tick_params(axis='both', labelsize=7)
    ax.set_ylim(-1.0, 1.0)

for ax in axes[-4:]:
    ax.set_xlabel('Coeff idx', fontsize=8)
for k in range(0, n_patches, 4):
    axes[k].set_ylabel('Value', fontsize=8)

fig.suptitle(f'Z coefficients per patch (32x36) — {VIZ_SCAN}', fontsize=14, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.985])

buf = BytesIO()
fig.savefig(buf, format='png', dpi=140, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

In [ ]:
plot_chamfer_heatmap(
    r['src_aligned'],
    r['morphed_ref'],
    title=f"{VIZ_SCAN} — aligned scan colored by distance to morphed ref",
)

In [ ]:
plot_chamfer_heatmap(
    r['morphed_ref'],
    r['mean_ref'],
    title=f"{VIZ_SCAN} — aligned scan colored by distance to morphed ref",
)